# Dimitris' Lyapunov analysis — read & unpack

Loads the MATLAB Lyapunov-exponent outputs in `outputs/dimitris_lyapunov/` and unpacks them into
tidy tables plus per-epoch summaries and plots.

**What the files contain.** Each `epoch_NN``Lexp.mat` holds a single variable `result` of shape
`(1, 1020)` — one **largest Lyapunov exponent per channel** (1020 ≈ the routed HD-MEA electrodes)
for that recording segment ("epoch"). Values are small and signed: λ > 0 indicates local trajectory
divergence (chaotic), λ < 0 convergence (stable), λ ≈ 0 the edge of chaos.

**Metadata still to confirm with Dimitris (not encoded in the files):**
1. **Epoch → segment.** Which recording / well / DIV and time window each epoch index refers to (only 01, 03, 05, 07, 10 are present).
2. **Channel order.** The order of the 1020 values — needed to map them onto electrode coordinates for a spatial map.
3. **Units & method.** Whether λ is per sample or per second, the embedding parameters (delay, dimension, Theiler window), and whether `result` is the largest exponent or one entry of a spectrum.

Until (1)–(3) are pinned down this notebook stays descriptive: it unpacks the arrays and characterises
their distributions, without asserting "chaos" from the raw magnitudes.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "outputs").exists():
    repo_root = repo_root.parent
LYAP_DIR = repo_root / "outputs" / "dimitris_lyapunov"
assert LYAP_DIR.exists(), f"Missing folder: {LYAP_DIR}"

files = sorted(LYAP_DIR.glob("epoch_*Lexp.mat"))
print(f"{len(files)} Lyapunov file(s):", [f.name for f in files])

# colourblind-safe categorical hues (Okabe-Ito), assigned to epochs in fixed sorted order (never cycled)
EPOCH_COLORS = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9", "#F0E442", "#999999"]

## Load and unpack

In [ ]:
def epoch_number(path):
    m = re.search(r"epoch_(\d+)", path.stem)
    return int(m.group(1)) if m else -1

wide = {}
records = []
for path in files:
    ep = epoch_number(path)
    result = np.asarray(sio.loadmat(path)["result"], dtype=float).ravel()
    wide[ep] = result
    records.extend({"epoch": ep, "channel_idx": ch, "lyapunov_exponent": float(v)}
                   for ch, v in enumerate(result))

epochs = sorted(wide)
lyap_wide = pd.DataFrame({ep: wide[ep] for ep in epochs})            # rows = channel, cols = epoch
lyap_long = pd.DataFrame.from_records(records).sort_values(["epoch", "channel_idx"]).reset_index(drop=True)

n_channels = {ep: int(np.isfinite(wide[ep]).sum()) for ep in epochs}
print(f"epochs present: {epochs}")
print(f"channels per epoch: {sorted(set(v.size for v in wide.values()))} (finite: {sorted(set(n_channels.values()))})")
missing = [e for e in range(min(epochs), max(epochs) + 1) if e not in epochs]
if missing:
    print(f"epoch indices NOT provided in this range: {missing}")

lyap_long.to_csv(LYAP_DIR / "lyapunov_tidy.csv", index=False)
print(f"Wrote tidy table: {LYAP_DIR / 'lyapunov_tidy.csv'}  ({len(lyap_long):,} rows)")
lyap_long.head()

## Per-epoch summary

In [ ]:
rows = []
for ep in epochs:
    v = lyap_wide[ep].to_numpy(float)
    v = v[np.isfinite(v)]
    pos = v[v > 0]
    rows.append({
        "epoch": ep,
        "n_channels": int(v.size),
        "mean": float(np.mean(v)),
        "median": float(np.median(v)),
        "std": float(np.std(v, ddof=1)),
        "min": float(np.min(v)),
        "max": float(np.max(v)),
        "frac_positive": float(np.mean(v > 0)),
        "mean_positive_only": float(np.mean(pos)) if pos.size else np.nan,
    })
lyap_summary = pd.DataFrame(rows)
lyap_summary.to_csv(LYAP_DIR / "lyapunov_epoch_summary.csv", index=False)
print(f"Wrote summary: {LYAP_DIR / 'lyapunov_epoch_summary.csv'}")
lyap_summary.round(6)

## Distribution of per-channel λ by epoch

Each violin is one epoch's 1020 per-channel exponents; the dashed line marks λ = 0 (the edge of chaos).
Epochs sit around zero — most split roughly evenly positive/negative, so the sign structure, not a large
positive bulk, is what distinguishes them.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
data = [lyap_wide[ep].to_numpy(float) for ep in epochs]
parts = ax.violinplot(data, showmedians=True, showextrema=False)
for i, body in enumerate(parts["bodies"]):
    body.set_facecolor(EPOCH_COLORS[i % len(EPOCH_COLORS)])
    body.set_edgecolor("0.35")
    body.set_alpha(0.65)
if "cmedians" in parts:
    parts["cmedians"].set_color("0.2")
ax.axhline(0.0, color="0.5", lw=1.0, ls="--", zorder=0, label="λ = 0 (edge of chaos)")
ax.set_xticks(range(1, len(epochs) + 1))
ax.set_xticklabels([f"ep {e:02d}" for e in epochs])
ax.set_xlabel("epoch (recording segment)")
ax.set_ylabel("largest Lyapunov exponent (per channel)")
ax.set_title("Per-channel largest Lyapunov exponent by epoch")
ax.legend(frameon=False, fontsize=8)
ax.grid(axis="y", color="0.9", lw=0.6, zorder=0)
fig.tight_layout()
fig.savefig(LYAP_DIR / "lyapunov_distributions.png", dpi=130)
plt.show()

## Polarity summary: chaotic fraction and mean λ per epoch

Left: the fraction of channels with λ > 0 (nominally "chaotic"), against the 50 % no-bias line.
Right: the mean exponent per epoch with a 95 % CI across channels, against λ = 0. These are the
headline per-epoch numbers — but interpret the sign, not the absolute magnitude, until the units and
embedding parameters are confirmed.

In [ ]:
x = np.arange(len(epochs))
labels = [f"ep {e:02d}" for e in epochs]
colors = [EPOCH_COLORS[i % len(EPOCH_COLORS)] for i in range(len(epochs))]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))

fp = lyap_summary["frac_positive"].to_numpy(float)
axes[0].bar(x, fp, color=colors, edgecolor="0.3", linewidth=0.6)
axes[0].axhline(0.5, color="0.5", ls="--", lw=1.0, label="50% (no sign bias)")
for xi, val in zip(x, fp):
    axes[0].text(xi, val + 0.015, f"{val:.0%}", ha="center", va="bottom", fontsize=8)
axes[0].set_ylim(0, 1)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("fraction of channels with λ > 0")
axes[0].set_title("Chaotic fraction per epoch")
axes[0].legend(frameon=False, fontsize=8)

mean = lyap_summary["mean"].to_numpy(float)
sem = (lyap_summary["std"] / np.sqrt(lyap_summary["n_channels"].clip(lower=1))).to_numpy(float)
axes[1].errorbar(x, mean, yerr=1.96 * sem, fmt="o", color="#0072B2", capsize=3, lw=1.4)
axes[1].axhline(0.0, color="0.5", ls="--", lw=1.0)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("mean largest Lyapunov exponent (±95% CI)")
axes[1].set_title("Mean λ per epoch")

fig.tight_layout()
fig.savefig(LYAP_DIR / "lyapunov_summary.png", dpi=130)
plt.show()

## Notes & next steps

- **Outputs written** to `outputs/dimitris_lyapunov/`: `lyapunov_tidy.csv` (epoch × channel long form),
  `lyapunov_epoch_summary.csv`, and the two figures.
- **Spatial map (pending).** Because 1020 matches the routed-electrode count, a diverging λ map over the
  array (centred at 0) would be informative — but only once Dimitris confirms the **channel ordering** so
  the values can be placed on electrode coordinates. Wiring that in is a one-cell add once the order is known.
- **Cross-referencing to the paper.** If the epochs correspond to the `stim_removal_null` segments used in
  Figures 2–4, the chaotic fraction / mean λ per epoch can be lined up against DIV, activity state, or the
  synchrony/anisotropy measures — but that needs the **epoch → segment** mapping first.